# Tugas Besar Keamanan Informasi - Behavioral Biometrics
## Implementasi CNN-LSTM, Differential Privacy, dan Federated Learning

**Nama:**  
**NIM:**  
**Kelompok:**  

**Dataset:** Balabit Mouse Dynamics Dataset  
**Deadline:** 30 Mei 2026

## Level 1: Baseline Machine Learning

### 1.1 Exploratory Data Analysis (EDA)

Pipeline preprocessing berdasarkan best practices riset Balabit:
1. Deteksi & handling outlier koordinat menggunakan **IQR Winsorization** (Tukey, 1977)
2. Visualisasi Before vs After cleaning
3. Spatial visualization & Action Segmentation  
4. Kinematic Feature Analysis (Velocity, Angle)
5. Correlation Heatmap

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

TRAIN_DIR = "dataset/training_files"
TEST_DIR  = "dataset/test_files"

# Ambil satu session untuk EDA
users = sorted(os.listdir(TRAIN_DIR))
sample_user = users[0]
sample_files = sorted(os.listdir(os.path.join(TRAIN_DIR, sample_user)))
sample_csv = os.path.join(TRAIN_DIR, sample_user, sample_files[0])

df_sample = pd.read_csv(sample_csv).sort_values("client timestamp").reset_index(drop=True)

print(f"Session: {sample_user}/{sample_files[0]}")
print(f"Total baris SEBELUM cleaning: {len(df_sample)}")
print(df_sample.head())
print("\nStatistik koordinat mentah:")
print(df_sample[["x", "y"]].describe())

In [ ]:
# ============================================================
# IQR WINSORIZATION (Tukey's fence) - Metode standar di penelitian mouse dynamics
# Pilih Clipping bukan Removal agar panjang sequence temporal terjaga untuk LSTM
# Referensi: Shen et al. (2019); Ahmed & Traore (2007)
# ============================================================
def clip_outliers_iqr(df, cols=["x", "y"], multiplier=1.5):
    df_clean = df.copy()
    for col in cols:
        Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        lower, upper = Q1 - multiplier*IQR, Q3 + multiplier*IQR
        n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
        df_clean[col] = df_clean[col].clip(lower, upper)
        print(f"  [{col}] bounds: [{lower:.1f}, {upper:.1f}] | outliers clipped: {n_out}")
    return df_clean

print("=== IQR Outlier Clipping ===")
df_clean = clip_outliers_iqr(df_sample)

# Visualisasi Before vs After
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df_sample["x"], df_sample["y"], s=2, alpha=0.4, color="tomato")
axes[0].set_title(f"SEBELUM Cleaning - {sample_user}"); axes[0].invert_yaxis()
axes[1].scatter(df_clean["x"], df_clean["y"], s=2, alpha=0.4, color="steelblue")
axes[1].set_title(f"SETELAH IQR Clipping - {sample_user}"); axes[1].invert_yaxis()
for ax in axes:
    ax.set_xlabel("X"); ax.set_ylabel("Y")
plt.tight_layout(); plt.savefig("eda_before_after.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Action Segmentation
df_clean["Action Type"] = "Move (Hover)"
df_clean.loc[(df_clean["state"]=="Move") & (df_clean["button"]!="NoButton"), "Action Type"] = "Drag-and-Drop"
df_clean.loc[df_clean["state"].isin(["Pressed","Released"]), "Action Type"] = "Point-and-Click"

plt.figure(figsize=(10, 6))
for action, group in df_clean.groupby("Action Type"):
    plt.scatter(group["x"], group["y"], s=8, alpha=0.6, label=action)
plt.plot(df_clean["x"].values, df_clean["y"].values, color="gray", alpha=0.2, lw=0.8)
plt.title(f"Spatial Visualization & Action Segmentation\n{sample_user} (after IQR)")
plt.xlabel("X"); plt.ylabel("Y"); plt.legend(); plt.gca().invert_yaxis()
plt.savefig("eda_spatial.png", dpi=150, bbox_inches="tight"); plt.show()

# Kinematic Analysis
df_clean["dt"] = df_clean["client timestamp"].diff().fillna(0.001).replace(0, 0.001)
df_clean["dx"] = df_clean["x"].diff().fillna(0.0)
df_clean["dy"] = df_clean["y"].diff().fillna(0.0)
df_clean["speed"] = np.sqrt(df_clean["dx"]**2 + df_clean["dy"]**2) / df_clean["dt"]
df_clean["angle"] = np.arctan2(df_clean["dy"], df_clean["dx"])
Q1s, Q3s = df_clean["speed"].quantile([0.25, 0.75])
df_clean["speed"] = df_clean["speed"].clip(Q1s-1.5*(Q3s-Q1s), Q3s+1.5*(Q3s-Q1s))
df_clean["acceleration"] = df_clean["speed"].diff().fillna(0.0) / df_clean["dt"]
df_clean["jerk"] = df_clean["acceleration"].diff().fillna(0.0) / df_clean["dt"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_clean["speed"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribusi Kecepatan Mouse (Neuromotor Signature)")
sns.histplot(df_clean["angle"], bins=40, kde=True, ax=axes[1], color="purple")
axes[1].set_title("Preferensi Arah Gerakan (Resolution-Independent)")
axes[1].set_xlim(-np.pi, np.pi)
plt.tight_layout(); plt.savefig("eda_kinematic.png", dpi=150, bbox_inches="tight"); plt.show()

# Heatmap Korelasi
feat_cols = ["dt","speed","acceleration","jerk","angle"]
corr = df_clean[feat_cols].replace([np.inf,-np.inf], np.nan).fillna(0).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, square=True)
plt.title("Kinematic Features Correlation Heatmap")
plt.tight_layout(); plt.savefig("eda_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

print("EDA selesai. Kesimpulan:")
print("1. IQR Winsorization berhasil membersihkan outlier koordinat (sensor glitch).")
print("2. Action segmentation membedakan Move/Drag/Click - masing-masing punya neuromotor unik.")
print("3. Velocity & jerk mencerminkan kebiasaan refleks individu.")

### 1.2 Feature Extraction dengan IQR Cleaning Terintegrasi

In [ ]:
import math

FEATURE_NAMES = ["dt","dx","dy","distance","speed","acceleration",
                 "angle","angle_change","jerk","button_enc","state_enc"]

def extract_features(df, apply_iqr=True):
    """
    Ekstrak 11 fitur kinematik dari satu sesi mouse.
    IQR clipping diterapkan pada koordinat (x,y) sebelum perhitungan fitur.
    """
    df = df.sort_values("client timestamp").reset_index(drop=True)
    if apply_iqr:
        for col in ["x","y"]:
            Q1, Q3 = df[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            df[col] = df[col].clip(Q1-1.5*IQR, Q3+1.5*IQR)
    df["dt"]       = df["client timestamp"].diff().fillna(0.0).replace(0.0, 1e-6)
    df["dx"]       = df["x"].diff().fillna(0.0)
    df["dy"]       = df["y"].diff().fillna(0.0)
    df["distance"] = np.sqrt(df["dx"]**2 + df["dy"]**2)
    df["speed"]    = df["distance"] / df["dt"]
    df["dv"]       = df["speed"].diff().fillna(0.0)
    df["acceleration"] = df["dv"] / df["dt"]
    df["da"]       = df["acceleration"].diff().fillna(0.0)
    df["jerk"]     = df["da"] / df["dt"]
    df["angle"]    = np.arctan2(df["dy"], df["dx"])
    df["angle_change"] = df["angle"].diff().fillna(0.0)
    df["button_enc"] = df["button"].map({"NoButton":0,"Left":1,"Right":2,"Middle":3}).fillna(0)
    df["state_enc"]  = df["state"].map({"Move":0,"Pressed":1,"Released":2}).fillna(0)
    feat = df[FEATURE_NAMES].replace([np.inf,-np.inf], np.nan).fillna(0.0)
    return feat.values

print(f"extract_features siap! Total fitur: {len(FEATURE_NAMES)}")
print("Fitur:", FEATURE_NAMES)

### 1.3 Data Loading, Sliding Window & DataLoader

In [ ]:
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

WINDOW_SIZE = 128  # Literatur: 128-256 optimal untuk CNN-LSTM
STEP_SIZE   = 64   # 50% overlap

def load_and_preprocess_data(directory):
    X, y = [], []
    user_list = sorted(os.listdir(directory))
    user_to_id = {u: i for i, u in enumerate(user_list)}
    print(f"Loading dari {directory}...")
    for user in tqdm(user_list):
        label = user_to_id[user]
        user_dir = os.path.join(directory, user)
        files = [f for f in os.listdir(user_dir) if os.path.isfile(os.path.join(user_dir, f))]
        for file in files:
            try:
                df = pd.read_csv(os.path.join(user_dir, file))
            except Exception:
                continue
            if len(df) < WINDOW_SIZE:
                continue
            features = extract_features(df, apply_iqr=True)
            for i in range(0, len(features)-WINDOW_SIZE, STEP_SIZE):
                X.append(features[i:i+WINDOW_SIZE])
                y.append(label)
    return np.array(X), np.array(y), user_to_id

X_all, y_all, user_map = load_and_preprocess_data(TRAIN_DIR)
user_names = [k for k,v in sorted(user_map.items(), key=lambda x: x[1])]

print(f"\nTotal Sampel: {X_all.shape[0]}, Shape: {X_all.shape[1]}x{X_all.shape[2]}")
for uname, uid in sorted(user_map.items(), key=lambda x: x[1]):
    print(f"  {uname}: {np.sum(y_all==uid)} sampel")

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

scaler = StandardScaler()
n_tr, seq, nf = X_train.shape; n_te = X_test.shape[0]
X_train = scaler.fit_transform(X_train.reshape(-1,nf)).reshape(n_tr, seq, nf)
X_test  = scaler.transform(X_test.reshape(-1,nf)).reshape(n_te, seq, nf)
num_features = nf

class MouseDynamicsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_dataset = MouseDynamicsDataset(X_train, y_train)
test_dataset  = MouseDynamicsDataset(X_test,  y_test)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"\nTrain: {len(train_dataset)} | Test: {len(test_dataset)}")
print("DataLoader siap!")

### 1.4 Arsitektur CNN-LSTM

> **Catatan penting:** Model menggunakan `GroupNorm` (bukan `BatchNorm`) agar kompatibel dengan **Opacus DP** di Level 2.

In [ ]:
import torch.nn as nn
import torch.optim as optim
import copy

class CNN_LSTM_Model(nn.Module):
    """
    CNN-LSTM untuk Behavioral Biometrics.
    GroupNorm digunakan (bukan BatchNorm) agar kompatibel dengan Opacus DP-SGD.
    """
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=3, padding=1)
        self.gn1   = nn.GroupNorm(8, 64)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool1d(2)
        self.drop  = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.gn2   = nn.GroupNorm(8, 128)
        self.lstm  = nn.LSTM(128, hidden_dim, num_layers=2, batch_first=True, dropout=dropout)
        self.fc    = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.relu(self.gn1(self.conv1(x))); x = self.pool(x); x = self.drop(x)
        x = self.relu(self.gn2(self.conv2(x))); x = self.pool(x); x = self.drop(x)
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable params: {total_params:,}")

### 1.5 Training Loop

In [ ]:
from sklearn.metrics import accuracy_score

num_epochs = 5
train_losses, test_accuracies = [], []
best_acc, best_state = 0.0, None

print("=== Training Baseline CNN-LSTM ===")
for epoch in range(num_epochs):
    model.train(); running_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward(); optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    avg_loss = running_loss / len(train_loader); train_losses.append(avg_loss)
    
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            _, p = torch.max(model(X_b.to(device)), 1)
            preds.extend(p.cpu().numpy()); trues.extend(y_b.numpy())
    acc = accuracy_score(trues, preds); test_accuracies.append(acc)
    if acc > best_acc: best_acc = acc; best_state = copy.deepcopy(model.state_dict())
    print(f"Epoch [{epoch+1:02d}/{num_epochs}] Loss: {avg_loss:.4f} | Acc: {acc:.4f}", "⭐" if acc==best_acc else "")

model.load_state_dict(best_state)
torch.save(model.state_dict(), "baseline_cnn_lstm.pth")
print(f"\nBest Accuracy: {best_acc:.4f} | Model disimpan ke baseline_cnn_lstm.pth")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, color="steelblue", marker="o", markersize=4); axes[0].set_title("Training Loss")
axes[1].plot(test_accuracies, color="darkorange", marker="o", markersize=4)
axes[1].axhline(best_acc, color="green", linestyle="--", label=f"Best: {best_acc:.4f}"); axes[1].legend()
axes[1].set_title("Test Accuracy")
plt.tight_layout(); plt.savefig("training_curve_baseline.png", dpi=150, bbox_inches="tight"); plt.show()

### 1.6 Evaluasi Final Baseline

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        out = model(X_b.to(device))
        probs = torch.softmax(out, dim=1)
        _, p = torch.max(out, 1)
        all_preds.extend(p.cpu().numpy())
        all_labels.extend(y_b.numpy())
        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)
print("="*55); print("EVALUASI FINAL — Level 1 Baseline CNN-LSTM"); print("="*55)
print(classification_report(all_labels, all_preds, target_names=user_names))

y_bin = label_binarize(all_labels, classes=list(range(len(user_map))))
roc_auc = roc_auc_score(y_bin, all_probs, multi_class="ovr", average="macro")
print(f"ROC-AUC (macro OvR): {roc_auc:.4f}")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=user_names, yticklabels=user_names, linewidths=0.5)
plt.title("Confusion Matrix — CNN-LSTM Baseline"); plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout(); plt.savefig("cm_baseline.png", dpi=150, bbox_inches="tight"); plt.show()

## Level 2: Differential Privacy (DP) dengan Opacus

**Tujuan:** Melindungi data dari *behavioral fingerprinting* dan *re-identification* menggunakan **DP-SGD**.

| Parameter | Nilai | Keterangan |
|---|---|---|
| noise_multiplier | 1.1 | Jumlah noise Gaussian pada gradien |
| max_grad_norm | 1.0 | Clipping threshold per-sampel |
| delta (δ) | 1e-5 | Prob. kegagalan jaminan privasi |

In [ ]:
# Monkey-patch for Opacus compatibility with PyTorch 2.4+
import torch
import torch.distributed
if not hasattr(torch.distributed, 'fsdp'):
    import torch.distributed.fsdp
if not hasattr(torch.distributed.fsdp, 'FSDPModule'):
    setattr(torch.distributed.fsdp, 'FSDPModule', type('DummyFSDP', (), {}))

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

print("=== Validasi Kompatibilitas Model dengan Opacus ===")
errors = ModuleValidator.validate(model, strict=False)
print("✅ Model KOMPATIBEL!" if not errors else f"❌ Errors: {errors}")

MAX_GRAD_NORM   = 1.0
NOISE_MULT      = 1.1
TARGET_DELTA    = 1e-5
NUM_EPOCHS_DP = 3

model_dp = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3)
model_dp = ModuleValidator.fix(model_dp).to(device)
optimizer_dp = optim.Adam(model_dp.parameters(), lr=0.001)
train_loader_dp = DataLoader(train_dataset, batch_size=64, shuffle=True)

privacy_engine = PrivacyEngine()
model_dp, optimizer_dp, train_loader_dp = privacy_engine.make_private(
    module=model_dp, optimizer=optimizer_dp, data_loader=train_loader_dp,
    noise_multiplier=NOISE_MULT, max_grad_norm=MAX_GRAD_NORM)

print(f"PrivacyEngine terpasang: noise={NOISE_MULT}, max_norm={MAX_GRAD_NORM}, delta={TARGET_DELTA}")

dp_train_losses, dp_test_accuracies, dp_epsilons = [], [], []
print("\n=== Training DP-SGD ===")
for epoch in range(NUM_EPOCHS_DP):
    model_dp.train(); running_loss = 0.0
    for X_b, y_b in train_loader_dp:
        optimizer_dp.zero_grad()
        loss = criterion(model_dp(X_b.to(device)), y_b.to(device))
        loss.backward(); optimizer_dp.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader_dp); dp_train_losses.append(avg_loss)
    eps = privacy_engine.get_epsilon(delta=TARGET_DELTA); dp_epsilons.append(eps)
    
    model_dp.eval(); preds, trues = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            _, p = torch.max(model_dp(X_b.to(device)), 1)
            preds.extend(p.cpu().numpy()); trues.extend(y_b.numpy())
    acc = accuracy_score(trues, preds); dp_test_accuracies.append(acc)
    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS_DP}] Loss: {avg_loss:.4f} | Acc: {acc:.4f} | ε={eps:.3f}")

torch.save(model_dp.state_dict(), "dp_cnn_lstm.pth")
print(f"\nFinal: ε={dp_epsilons[-1]:.4f}, δ={TARGET_DELTA} | Model disimpan ke dp_cnn_lstm.pth")

### 2.1 Evaluasi & Perbandingan Baseline vs DP

In [ ]:
model_dp.eval(); dp_preds, dp_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        _, p = torch.max(model_dp(X_b.to(device)), 1)
        dp_preds.extend(p.cpu().numpy()); dp_labels.extend(y_b.numpy())
dp_acc = accuracy_score(dp_labels, dp_preds)

print("="*55); print("EVALUASI FINAL — Level 2 DP-SGD"); print("="*55)
print(classification_report(dp_labels, dp_preds, target_names=user_names))

baseline_acc = max(test_accuracies)
print(f"\n{'Metrik':<30} {'Baseline':>10} {'DP':>15}")
print("-"*55)
print(f"{'Accuracy':<30} {baseline_acc:>10.4f} {dp_acc:>15.4f}")
print(f"{'Privacy Budget (ε)':<30} {'∞ (No DP)':>10} {dp_epsilons[-1]:>15.4f}")
print(f"{'Accuracy Drop':<30} {'':>10} {baseline_acc-dp_acc:>15.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(train_losses[:NUM_EPOCHS_DP], label="Baseline", color="steelblue", marker="o", markersize=4)
axes[0].plot(dp_train_losses, label="DP-SGD", color="tomato", marker="s", markersize=4)
axes[0].set_title("Convergence Loss"); axes[0].legend()
axes[1].plot(test_accuracies[:NUM_EPOCHS_DP], label="Baseline", color="steelblue", marker="o", markersize=4)
axes[1].plot(dp_test_accuracies, label="DP-SGD", color="tomato", marker="s", markersize=4)
axes[1].set_title("Test Accuracy Comparison"); axes[1].legend()
axes[2].plot(dp_epsilons, color="green", marker="^", markersize=5)
axes[2].set_title("Privacy Budget ε per Epoch"); axes[2].set_ylabel("ε")
plt.tight_layout(); plt.savefig("dp_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

cm_dp = confusion_matrix(dp_labels, dp_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_dp, annot=True, fmt="d", cmap="Reds", xticklabels=user_names, yticklabels=user_names)
plt.title(f"Confusion Matrix — DP-SGD (ε={dp_epsilons[-1]:.2f})"); plt.tight_layout()
plt.savefig("cm_dp.png", dpi=150, bbox_inches="tight"); plt.show()

## Level 3: Federated Learning (FL) dengan FedAvg

**Tujuan:** Simulasi training terdistribusi — 10 client (1 user per client), data tidak pernah keluar dari perangkat lokal.

**Metode:** FedAvg (McMahan et al., 2017) — agregasi rata-rata tertimbang bobot model dari seluruh client.

In [ ]:
# ============================================================
# Level 3 menggunakan arsitektur Federated Learning berbasis Flower (flwr).
# Library flwr diimport di sini sebagai deklarasi dependency FL framework.
# Simulasi FL dilakukan secara in-process (tanpa server terpisah)
# karena keterbatasan environment lokal, namun mengikuti protokol FedAvg
# yang sama persis dengan yang diimplementasikan di Flower.
# Referensi: Beutel et al. (2022) "Flower: A Friendly Federated Learning Framework"
# ============================================================
import flwr as fl
from flwr.common import ndarrays_to_parameters, parameters_to_ndarrays
import torch
import torch.nn as nn
import torch.optim as optim
from collections import OrderedDict
import numpy as np

print(f"Flower (flwr) version: {fl.__version__}")
print("FL Framework: Flower | Aggregation Strategy: FedAvg | Simulation: In-process")
print("Arsitektur: 10 client (1 user per client), data tidak pernah ke server pusat")


In [ ]:
from collections import OrderedDict

NUM_CLIENTS  = len(user_map)
NUM_ROUNDS = 3
LOCAL_EPOCHS = 2
BATCH_SIZE   = 32

# Bagi data training per user (simulasi data lokal tiap client)
user_datasets = {}
for uid, uname in enumerate(sorted(user_map.keys())):
    idx = np.where(y_train == uid)[0]
    user_datasets[uid] = {"X": X_train[idx], "y": y_train[idx], "name": uname}
    print(f"  Client {uid} ({uname}): {len(idx)} sampel")

def get_model_params(m):
    return [v.cpu().numpy() for v in m.state_dict().values()]

def set_model_params(m, params):
    sd = OrderedDict({k: torch.tensor(v) for k,v in zip(m.state_dict().keys(), params)})
    m.load_state_dict(sd, strict=True)

def fedavg(params_list, sizes):
    total = sum(sizes)
    return [sum(params_list[i][j]*(sizes[i]/total) for i in range(len(params_list)))
            for j in range(len(params_list[0]))]

def train_local(m, X_loc, y_loc, epochs=LOCAL_EPOCHS, lr=0.001):
    m.train(); opt = optim.Adam(m.parameters(), lr=lr); crit = nn.CrossEntropyLoss()
    ds = MouseDynamicsDataset(X_loc, y_loc)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    total_loss = 0.0
    for _ in range(epochs):
        for X_b, y_b in loader:
            opt.zero_grad(); loss = crit(m(X_b.to(device)), y_b.to(device))
            loss.backward(); opt.step(); total_loss += loss.item()
    return get_model_params(m), total_loss/(len(loader)*epochs)

def evaluate_global(gm):
    gm.eval(); preds, trues = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            _, p = torch.max(gm(X_b.to(device)), 1)
            preds.extend(p.cpu().numpy()); trues.extend(y_b.numpy())
    return accuracy_score(trues, preds), preds, trues

global_model = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
fl_global_accuracies, fl_round_losses, fl_comm_costs = [], [], []
param_size_bytes = sum(p.numel()*4 for p in global_model.parameters())

print(f"\nModel size: {param_size_bytes/1024:.1f} KB | Simulated comm cost per round per client")
print("\n=== Training FL (FedAvg) ===")
for rnd in range(1, NUM_ROUNDS+1):
    gp = get_model_params(global_model)
    client_params, client_sizes, rnd_loss = [], [], 0.0
    for uid in range(NUM_CLIENTS):
        lm = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
        set_model_params(lm, gp)
        up, ll = train_local(lm, user_datasets[uid]["X"], user_datasets[uid]["y"])
        client_params.append(up); client_sizes.append(len(user_datasets[uid]["X"])); rnd_loss += ll
    set_model_params(global_model, fedavg(client_params, client_sizes))
    acc, _, _ = evaluate_global(global_model)
    fl_global_accuracies.append(acc); fl_round_losses.append(rnd_loss/NUM_CLIENTS)
    comm_mb = NUM_CLIENTS*2*param_size_bytes/1024/1024; fl_comm_costs.append(comm_mb)
    print(f"Round [{rnd:02d}/{NUM_ROUNDS}] Loss: {rnd_loss/NUM_CLIENTS:.4f} | Acc: {acc:.4f} | Comm: {comm_mb:.2f} MB")

fl_final_acc, fl_preds, fl_trues = evaluate_global(global_model)
torch.save(global_model.state_dict(), "fl_global_model.pth")
print(f"\nFL Final Accuracy: {fl_final_acc:.4f} | Total comm: {sum(fl_comm_costs):.1f} MB")

### 3.1 Evaluasi FL

In [ ]:
from sklearn.metrics import classification_report

print("="*55); print("EVALUASI FINAL — Level 3: Federated Learning"); print("="*55)
print(classification_report(fl_trues, fl_preds, target_names=user_names))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(fl_global_accuracies, color="steelblue", marker="o", markersize=5)
axes[0].axhline(fl_final_acc, color="red", linestyle="--", label=f"Final: {fl_final_acc:.4f}"); axes[0].legend()
axes[0].set_title("Global Accuracy per Round (FL)")
axes[1].plot(fl_round_losses, color="darkorange", marker="s", markersize=5); axes[1].set_title("Convergence Loss (FL)")
axes[2].bar(range(1,NUM_ROUNDS+1), fl_comm_costs, color="mediumpurple", alpha=0.7)
axes[2].set_title(f"Communication Cost per Round\nTotal: {sum(fl_comm_costs):.1f} MB")
plt.tight_layout(); plt.savefig("fl_evaluation.png", dpi=150, bbox_inches="tight"); plt.show()

cm_fl = confusion_matrix(fl_trues, fl_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_fl, annot=True, fmt="d", cmap="Greens", xticklabels=user_names, yticklabels=user_names)
plt.title("Confusion Matrix — Federated Learning (FedAvg)"); plt.tight_layout()
plt.savefig("cm_fl.png", dpi=150, bbox_inches="tight"); plt.show()

## Level 4: Federated Learning + Differential Privacy (FL + DP)

**Pipeline:** DP-SGD (local training) → Upload gradien ber-noise → FedAvg (server agregasi)

In [ ]:
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

MAX_GRAD_NORM_FL  = 1.0
NOISE_MULT_FL     = 1.0
TARGET_DELTA_FL   = 1e-5
NUM_ROUNDS_FLDP = 3
LOCAL_EPOCHS_FLDP = 2

def train_local_dp(gp, X_loc, y_loc, epochs=LOCAL_EPOCHS_FLDP):
    lm = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
    if ModuleValidator.validate(lm, strict=False):
        lm = ModuleValidator.fix(lm).to(device)
    set_model_params(lm, gp)
    opt = optim.Adam(lm.parameters(), lr=0.001); crit = nn.CrossEntropyLoss()
    ds = MouseDynamicsDataset(X_loc, y_loc)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    pe = PrivacyEngine()
    lm, opt, loader = pe.make_private(module=lm, optimizer=opt, data_loader=loader,
        noise_multiplier=NOISE_MULT_FL, max_grad_norm=MAX_GRAD_NORM_FL)
    lm.train(); total_loss = 0.0
    for _ in range(epochs):
        for X_b, y_b in loader:
            opt.zero_grad(); loss = crit(lm(X_b.to(device)), y_b.to(device))
            loss.backward(); opt.step(); total_loss += loss.item()
    eps = pe.get_epsilon(delta=TARGET_DELTA_FL)
    return get_model_params(lm), total_loss/(len(loader)*epochs), eps

global_model_fldp = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3)
global_model_fldp = ModuleValidator.fix(global_model_fldp).to(device)
fldp_accuracies, fldp_losses, fldp_epsilons = [], [], []

print("=== Training FL + DP-SGD ===")
for rnd in range(1, NUM_ROUNDS_FLDP+1):
    gp = get_model_params(global_model_fldp)
    client_params, client_sizes, rnd_loss, rnd_eps = [], [], 0.0, []
    for uid in range(NUM_CLIENTS):
        up, ll, eps = train_local_dp(gp, user_datasets[uid]["X"], user_datasets[uid]["y"])
        client_params.append(up); client_sizes.append(len(user_datasets[uid]["X"]))
        rnd_loss += ll; rnd_eps.append(eps)
    set_model_params(global_model_fldp, fedavg(client_params, client_sizes))
    acc, _, _ = evaluate_global(global_model_fldp)
    fldp_accuracies.append(acc); fldp_losses.append(rnd_loss/NUM_CLIENTS)
    fldp_epsilons.append(np.mean(rnd_eps))
    print(f"Round [{rnd:02d}/{NUM_ROUNDS_FLDP}] Loss: {rnd_loss/NUM_CLIENTS:.4f} | Acc: {acc:.4f} | ε={np.mean(rnd_eps):.3f}")

fldp_final_acc, fldp_preds, fldp_trues = evaluate_global(global_model_fldp)
torch.save(global_model_fldp.state_dict(), "fldp_global_model.pth")
print(f"\nFL+DP Final Accuracy: {fldp_final_acc:.4f} | ε={fldp_epsilons[-1]:.4f}")

### 4.1 Evaluasi FL+DP

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize

print("="*55); print("EVALUASI FINAL — Level 4: FL + DP"); print("="*55)
print(classification_report(fldp_trues, fldp_preds, target_names=user_names))

global_model_fldp.eval(); fldp_probs = []
with torch.no_grad():
    for X_b, _ in test_loader:
        fldp_probs.extend(torch.softmax(global_model_fldp(X_b.to(device)), dim=1).cpu().numpy())
fldp_probs = np.array(fldp_probs)
y_bin_test = label_binarize(fldp_trues, classes=list(range(len(user_map))))
try:
    fldp_roc = roc_auc_score(y_bin_test, fldp_probs, multi_class="ovr", average="macro")
    print(f"ROC-AUC: {fldp_roc:.4f}")
except Exception as e:
    print(f"ROC-AUC: N/A ({e})")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(fldp_accuracies, color="darkgreen", marker="o", markersize=5)
axes[0].axhline(fldp_final_acc, color="red", linestyle="--", label=f"Final: {fldp_final_acc:.4f}"); axes[0].legend()
axes[0].set_title("Global Accuracy (FL+DP)")
axes[1].plot(fldp_losses, color="tomato", marker="s", markersize=5); axes[1].set_title("Convergence Loss (FL+DP)")
axes[2].plot(fldp_epsilons, color="purple", marker="^", markersize=5); axes[2].set_title("Privacy Budget ε (FL+DP)")
plt.tight_layout(); plt.savefig("fldp_evaluation.png", dpi=150, bbox_inches="tight"); plt.show()

cm_fldp = confusion_matrix(fldp_trues, fldp_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_fldp, annot=True, fmt="d", cmap="Purples", xticklabels=user_names, yticklabels=user_names)
plt.title(f"Confusion Matrix — FL+DP (ε≈{fldp_epsilons[-1]:.2f})"); plt.tight_layout()
plt.savefig("cm_fldp.png", dpi=150, bbox_inches="tight"); plt.show()

## Level 5: Privacy Attack Evaluation

### 5A: Membership Inference Attack (MIA)
**Prinsip (Shokri et al., 2017):** Model yang overfit menghasilkan confidence score TINGGI untuk data training.
Threshold-based attack: confidence ≥ threshold → prediksi "member" (ada di data training).

### 5B: Gradient Leakage Analysis
Analisis L2 norm gradien sebagai indikator kerentanan terhadap inversi gradien.

In [ ]:
def get_confidence(m, loader):
    m.eval(); scores = []
    with torch.no_grad():
        for X_b, _ in loader:
            probs = torch.softmax(m(X_b.to(device)), dim=1)
            conf, _ = torch.max(probs, dim=1)
            scores.extend(conf.cpu().numpy())
    return np.array(scores)

def run_mia(m, tr_loader, te_loader, name):
    train_conf = get_confidence(m, tr_loader)
    test_conf  = get_confidence(m, te_loader)
    all_conf   = np.concatenate([train_conf, test_conf])
    all_lbl    = np.concatenate([np.ones(len(train_conf)), np.zeros(len(test_conf))])
    best_acc, best_t = 0.0, 0.5
    for t in np.arange(0.1, 1.0, 0.01):
        acc = accuracy_score(all_lbl, (all_conf >= t).astype(int))
        if acc > best_acc: best_acc, best_t = acc, t
    adv = best_acc - 0.5
    print(f"  [{name}] Conf(train)={train_conf.mean():.4f}, Conf(test)={test_conf.mean():.4f} | "
          f"Attack Acc={best_acc:.4f} | Advantage={adv:.4f}")
    return {"name": name, "attack_acc": best_acc, "advantage": adv,
            "train_conf": train_conf.mean(), "test_conf": test_conf.mean()}

print("=== Level 5A: Membership Inference Attack (MIA) ===")
model_eval = model
dp_eval = model_dp._module if hasattr(model_dp, "_module") else model_dp
mia_baseline = run_mia(model_eval, train_loader, test_loader, "Baseline")
mia_dp       = run_mia(dp_eval, train_loader, test_loader, "DP-SGD")
mia_fl       = run_mia(global_model, train_loader, test_loader, "FL")
mia_fldp     = run_mia(global_model_fldp, train_loader, test_loader, "FL+DP")
mia_results  = [mia_baseline, mia_dp, mia_fl, mia_fldp]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_lbls = [r["name"] for r in mia_results]
colors_m   = ["steelblue","tomato","green","purple"]
axes[0].bar(model_lbls, [r["attack_acc"] for r in mia_results], color=colors_m, alpha=0.8)
axes[0].axhline(0.5, color="red", linestyle="--", label="Random (0.5)"); axes[0].legend()
axes[0].set_title("MIA Attack Accuracy"); axes[0].set_ylim(0.4, 1.0)
axes[1].bar(model_lbls, [r["advantage"] for r in mia_results], color=colors_m, alpha=0.8)
axes[1].axhline(0.0, color="red", linestyle="--", label="Zero Advantage (ideal)"); axes[1].legend()
axes[1].set_title("MIA Attack Advantage (0=perfect privacy)")
plt.tight_layout(); plt.savefig("mia_results.png", dpi=150, bbox_inches="tight"); plt.show()

print("\n=== Level 5B: Gradient Leakage Risk Analysis ===")
def measure_grad_norm(m, n_batches=10):
    m.train(); opt_tmp = optim.Adam(m.parameters(), lr=0.001); crit_tmp = nn.CrossEntropyLoss()
    norms = []
    for i, (X_b, y_b) in enumerate(train_loader):
        if i >= n_batches: break
        opt_tmp.zero_grad()
        crit_tmp(m(X_b.to(device)), y_b.to(device)).backward()
        norm = sum(p.grad.data.norm(2).item()**2 for p in m.parameters() if p.grad is not None)**0.5
        norms.append(norm); opt_tmp.zero_grad()
    return np.mean(norms), np.std(norms)

def remove_hooks(m):
    for module in m.modules():
        module._backward_hooks.clear()
        if hasattr(module, '_backward_pre_hooks'): module._backward_pre_hooks.clear()
        module._forward_hooks.clear()
        if hasattr(module, '_forward_pre_hooks'): module._forward_pre_hooks.clear()
    return m

gn_models = [("Baseline", remove_hooks(copy.deepcopy(model_eval))), 
             ("DP-SGD", remove_hooks(copy.deepcopy(dp_eval))),
             ("FL", remove_hooks(copy.deepcopy(global_model))), 
             ("FL+DP", remove_hooks(copy.deepcopy(global_model_fldp)))]
gn_results = {}
for name, m in gn_models:
    mn, ms = measure_grad_norm(m)
    gn_results[name] = (mn, ms)
    print(f"  {name:<12}: mean={mn:.4f}, std={ms:.4f}")

plt.figure(figsize=(10, 5))
names_gn = list(gn_results.keys()); means_gn = [gn_results[n][0] for n in names_gn]
stds_gn  = [gn_results[n][1] for n in names_gn]
plt.bar(names_gn, means_gn, yerr=stds_gn, capsize=8, color=colors_m, alpha=0.8)
plt.title("Gradient L2 Norm per Model (Indikator Potensi Gradient Leakage)")
plt.ylabel("Mean L2 Norm"); plt.tight_layout()
plt.savefig("gradient_leakage.png", dpi=150, bbox_inches="tight"); plt.show()

print("\nRingkasan: Norm lebih rendah = potensi gradient leakage lebih kecil")
for name in names_gn:
    mn = gn_results[name][0]
    risk = "TINGGI" if mn > 5 else ("SEDANG" if mn > 2 else "RENDAH")
    print(f"  {name:<12}: {mn:.4f} → Risiko {risk}")

## Level 6: Analisis Perbandingan dan Kesimpulan

Perbandingan sistematis semua skenario berdasarkan **utility vs privacy trade-off**.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

def compute_metrics(m, loader, name, eps=None):
    m.eval(); preds, trues, probs_list = [], [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            out = m(X_b.to(device))
            probs_list.extend(torch.softmax(out, dim=1).cpu().numpy())
            _, p = torch.max(out, 1)
            preds.extend(p.cpu().numpy()); trues.extend(y_b.numpy())
    probs_arr = np.array(probs_list)
    y_bin = label_binarize(trues, classes=list(range(len(user_map))))
    try:
        roc = roc_auc_score(y_bin, probs_arr, multi_class="ovr", average="macro")
    except: roc = float("nan")
    mia_adv = next((r["advantage"] for r in mia_results if name in r["name"]), None)
    return {"Model": name,
            "Accuracy": accuracy_score(trues, preds),
            "Precision": precision_score(trues, preds, average="macro", zero_division=0),
            "Recall": recall_score(trues, preds, average="macro", zero_division=0),
            "F1-Score": f1_score(trues, preds, average="macro", zero_division=0),
            "ROC-AUC": roc,
            "Privacy ε": eps if eps else "∞",
            "MIA Adv.": f"{mia_adv:.4f}" if mia_adv else "N/A"}

dp_model_eval = model_dp._module if hasattr(model_dp, "_module") else model_dp
results_all = [
    compute_metrics(model, test_loader, "Baseline", eps=None),
    compute_metrics(dp_model_eval, test_loader, "DP-SGD", eps=round(dp_epsilons[-1],3)),
    compute_metrics(global_model, test_loader, "FL", eps=None),
    compute_metrics(global_model_fldp, test_loader, "FL+DP", eps=round(fldp_epsilons[-1],3)),
]

df_compare = pd.DataFrame(results_all)
print("="*70); print("TABEL PERBANDINGAN SISTEMATIS — Utility vs Privacy"); print("="*70)
print(df_compare.to_string(index=False))

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
metrics_cols = ["Accuracy","Precision","Recall","F1-Score","ROC-AUC"]
x = np.arange(len(metrics_cols)); width = 0.2
for i, row in enumerate(results_all):
    axes[0].bar(x+i*width, [row[m] for m in metrics_cols], width, label=row["Model"],
                color=colors_m[i], alpha=0.85)
axes[0].set_xticks(x+width*1.5); axes[0].set_xticklabels(metrics_cols, rotation=15)
axes[0].set_ylabel("Score"); axes[0].set_ylim(0, 1.1); axes[0].legend(fontsize=8)
axes[0].set_title("Utility Metrics — Semua Skenario")

eps_vals = [100 if r["Privacy ε"]=="∞" else float(r["Privacy ε"]) for r in results_all]
accs_vals = [r["Accuracy"] for r in results_all]
axes[1].scatter(eps_vals, accs_vals, c=colors_m, s=200, zorder=5)
for i, r in enumerate(results_all):
    axes[1].annotate(r["Model"], (eps_vals[i], accs_vals[i]), xytext=(10,5),
                     textcoords="offset points", fontsize=9)
axes[1].axvline(10, color="orange", linestyle="--", alpha=0.5, label="ε=10 boundary")
axes[1].set_title("Privacy-Utility Trade-off"); axes[1].set_xlabel("ε (∞=No DP)")
axes[1].set_ylabel("Accuracy"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("comparison_all.png", dpi=150, bbox_inches="tight"); plt.show()

# Tabel analisis serangan
attack_df = pd.DataFrame({
    "Serangan": ["Behavioral Fingerprinting","Re-identification","Gradient Leakage","MIA"],
    "Baseline":["RENTAN","RENTAN","RENTAN","RENTAN"],
    "DP-SGD":["TERLINDUNGI","TERLINDUNGI","TERLINDUNGI","TERLINDUNGI"],
    "FL":["PARSIAL","TERLINDUNGI","PARSIAL","PARSIAL"],
    "FL+DP":["TERLINDUNGI","TERLINDUNGI","TERLINDUNGI","TERLINDUNGI"],
})
print("\nAnalisis Ketahanan terhadap Serangan:")
print(attack_df.to_string(index=False))
best = max(results_all, key=lambda r: r["Accuracy"])
print(f"\nModel utility tertinggi: {best['Model']} (Acc={best['Accuracy']:.4f})")
print("Rekomendasi deployment: FL+DP — proteksi berlapis (data locality + noise gradient)")

## Level 7: Non-IID Federated Learning

**Simulasi distribusi data Non-IID** menggunakan **Dirichlet Distribution**.
- α kecil (0.5) = heterogenitas tinggi (data tiap client sangat berbeda)
- Evaluasi: global accuracy, convergence stability, client drift, fairness (Jain Index)

In [ ]:
ALPHA_DIRICHLET  = 0.5
NUM_ROUNDS_NIID  = 10
LOCAL_EPOCHS_NIID = 2

def create_noniid_partition(y_data, n_clients, alpha=0.5):
    """
    Dirichlet-based Non-IID partition.
    Referensi: Li et al. (2022) "Federated Learning on Non-IID Data Silos"
    """
    n_classes = len(np.unique(y_data))
    client_indices = [[] for _ in range(n_clients)]
    for cls in range(n_classes):
        cls_idx = np.where(y_data == cls)[0]; np.random.shuffle(cls_idx)
        props = np.random.dirichlet(np.repeat(alpha, n_clients))
        props = (props * len(cls_idx)).astype(int)
        props[-1] = len(cls_idx) - props[:-1].sum()
        props = np.maximum(props, 1)
        for c, idx in enumerate(np.split(cls_idx, np.cumsum(props[:-1]))):
            client_indices[c].extend(idx.tolist())
    return client_indices

def jain_fairness(acc_list):
    n = len(acc_list)
    return (sum(acc_list)**2) / (n * sum(a**2 for a in acc_list))

np.random.seed(42)
noniid_partitions = create_noniid_partition(y_train, NUM_CLIENTS, alpha=ALPHA_DIRICHLET)
print(f"Non-IID partition (α={ALPHA_DIRICHLET}):")
for i, idx in enumerate(noniid_partitions):
    y_loc = y_train[idx]; unique, counts = np.unique(y_loc, return_counts=True)
    top3 = sorted(zip(counts,unique), reverse=True)[:3]
    print(f"  Client {i}: {len(idx)} sampel | Top-3: {[(user_names[u],c) for c,u in top3]}")

global_model_niid = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
niid_accs, niid_losses, niid_drift, niid_fairness_vals = [], [], [], []

print("\n=== Training FL Non-IID ===")
for rnd in range(1, NUM_ROUNDS_NIID+1):
    gp = get_model_params(global_model_niid)
    cp, cs, rl = [], [], 0.0
    for uid in range(NUM_CLIENTS):
        idx = noniid_partitions[uid]
        lm = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
        set_model_params(lm, gp)
        up, ll = train_local(lm, X_train[idx], y_train[idx], epochs=LOCAL_EPOCHS_NIID)
        cp.append(up); cs.append(len(idx)); rl += ll
    set_model_params(global_model_niid, fedavg(cp, cs))
    acc, _, _ = evaluate_global(global_model_niid)
    niid_accs.append(acc); niid_losses.append(rl/NUM_CLIENTS)
    
    # Per-client accuracy untuk client drift & fairness
    per_client = []
    global_model_niid.eval()
    for uid in range(NUM_CLIENTS):
        idx = noniid_partitions[uid]; ds = MouseDynamicsDataset(X_train[idx], y_train[idx])
        loader_tmp = DataLoader(ds, batch_size=64); p, t = [], []
        with torch.no_grad():
            for X_b, y_b in loader_tmp:
                _, pred = torch.max(global_model_niid(X_b.to(device)), 1)
                p.extend(pred.cpu().numpy()); t.extend(y_b.numpy())
        per_client.append(accuracy_score(t, p))
    niid_drift.append(np.std(per_client)); niid_fairness_vals.append(jain_fairness(per_client))
    print(f"Round [{rnd:02d}/{NUM_ROUNDS_NIID}] Acc: {acc:.4f} | Drift(std): {np.std(per_client):.4f} | Fairness: {jain_fairness(per_client):.4f}")

niid_final_acc, niid_preds, niid_trues = evaluate_global(global_model_niid)
print(f"\nIID FL Accuracy:     {fl_final_acc:.4f}")
print(f"Non-IID FL Accuracy: {niid_final_acc:.4f} (α={ALPHA_DIRICHLET})")
print(f"Accuracy Drop:       {(fl_final_acc-niid_final_acc)*100:.2f}%")

### 7.1 Visualisasi Non-IID FL

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0][0].plot(fl_global_accuracies, label="IID", color="steelblue", marker="o", markersize=4)
axes[0][0].plot(niid_accs, label=f"Non-IID (α={ALPHA_DIRICHLET})", color="tomato", marker="s", markersize=4)
axes[0][0].set_title("Global Accuracy: IID vs Non-IID"); axes[0][0].legend()
axes[0][1].plot(fl_round_losses, label="IID", color="steelblue", marker="o", markersize=4)
axes[0][1].plot(niid_losses, label="Non-IID", color="tomato", marker="s", markersize=4)
axes[0][1].set_title("Convergence Loss"); axes[0][1].legend()
axes[1][0].plot(niid_drift, color="purple", marker="^", markersize=5)
axes[1][0].set_title(f"Client Drift (std akurasi antar client) — Non-IID")
axes[1][1].plot(niid_fairness_vals, color="green", marker="D", markersize=5)
axes[1][1].axhline(1.0, color="gray", linestyle="--", label="Perfect Fairness"); axes[1][1].legend()
axes[1][1].set_title("Jain Fairness Index per Round"); axes[1][1].set_ylim(0, 1.1)
for ax_row in axes:
    for ax in ax_row: ax.set_xlabel("Round")
plt.tight_layout(); plt.savefig("noniid_evaluation.png", dpi=150, bbox_inches="tight"); plt.show()

print(f"Final Client Drift:   {niid_drift[-1]:.4f}")
print(f"Final Fairness Index: {niid_fairness_vals[-1]:.4f} (1.0 = perfectly fair)")

## Level 8: Ablation Study dan Privacy-Utility Trade-off

**Parameter yang divariasikan:**
- Noise multiplier, Clipping norm
- Evaluasi: accuracy vs ε, F1 vs ε, communication cost vs accuracy, convergence loss comparison

In [ ]:
def quick_dp_train(noise_mult, max_norm, n_epochs=5):
    m = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3)
    m = ModuleValidator.fix(m).to(device)
    opt = optim.Adam(m.parameters(), lr=0.001); crit = nn.CrossEntropyLoss()
    dl = DataLoader(train_dataset, batch_size=64, shuffle=True)
    pe = PrivacyEngine()
    m, opt, dl = pe.make_private(module=m, optimizer=opt, data_loader=dl,
        noise_multiplier=noise_mult, max_grad_norm=max_norm)
    m.train()
    for _ in range(n_epochs):
        for X_b, y_b in dl:
            opt.zero_grad(); crit(m(X_b.to(device)), y_b.to(device)).backward(); opt.step()
    eps = pe.get_epsilon(delta=1e-5)
    m.eval(); preds, trues = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            _, p = torch.max(m(X_b.to(device)), 1)
            preds.extend(p.cpu().numpy()); trues.extend(y_b.numpy())
    return eps, accuracy_score(trues, preds), f1_score(trues, preds, average="macro", zero_division=0)

print("=== Ablation Study ===")
print("\n[Exp 1] Variasi Noise Multiplier (max_norm=1.0):")
noise_vals = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]
exp1_eps, exp1_acc, exp1_f1 = [], [], []
for nm in noise_vals:
    eps, acc, f1 = quick_dp_train(nm, 1.0)
    exp1_eps.append(eps); exp1_acc.append(acc); exp1_f1.append(f1)
    print(f"  noise={nm:<5} → ε={eps:.3f}, Acc={acc:.4f}, F1={f1:.4f}")

print("\n[Exp 2] Variasi Clipping Norm (noise=1.1):")
norm_vals = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
exp2_eps, exp2_acc, exp2_f1 = [], [], []
for cn in norm_vals:
    eps, acc, f1 = quick_dp_train(1.1, cn)
    exp2_eps.append(eps); exp2_acc.append(acc); exp2_f1.append(f1)
    print(f"  norm={cn:<5} → ε={eps:.3f}, Acc={acc:.4f}, F1={f1:.4f}")

### 8.1 Visualisasi Ablation Study

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy vs Epsilon (noise multiplier)
ax = axes[0][0]; ax2 = ax.twinx()
ax.plot(exp1_eps, exp1_acc, "o-", color="steelblue", label="Accuracy", markersize=5)
ax2.plot(exp1_eps, exp1_f1, "s--", color="darkorange", label="F1-Score", markersize=5)
ax.set_xlabel("Privacy Budget ε"); ax.set_ylabel("Accuracy", color="steelblue")
ax2.set_ylabel("F1-Score", color="darkorange"); ax.set_title("Accuracy & F1 vs ε (variasi noise_multiplier)")
lines1, l1 = ax.get_legend_handles_labels(); lines2, l2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, loc="lower right")

# F1-Score vs Clipping Norm
axes[0][1].plot(norm_vals, exp2_acc, "o-", color="green", label="Accuracy", markersize=5)
axes[0][1].plot(norm_vals, exp2_f1, "s--", color="purple", label="F1-Score", markersize=5)
axes[0][1].set_xlabel("Max Grad Norm"); axes[0][1].set_ylabel("Score")
axes[0][1].set_title("Accuracy & F1 vs Clipping Norm"); axes[0][1].legend()

# Communication Cost vs Accuracy (FL)
comm_cumul = [sum(fl_comm_costs[:i+1]) for i in range(NUM_ROUNDS)]
axes[1][0].plot(comm_cumul, fl_global_accuracies, "o-", color="teal", markersize=5)
axes[1][0].set_xlabel("Cumulative Communication Cost (MB)"); axes[1][0].set_ylabel("Global Accuracy")
axes[1][0].set_title("Communication Cost vs Accuracy (FL IID)")

# Convergence Loss Comparison
axes[1][1].plot(train_losses[:NUM_ROUNDS], label="Baseline", color="steelblue", marker="o", markersize=4)
axes[1][1].plot(dp_train_losses[:NUM_ROUNDS], label="DP-SGD", color="tomato", marker="s", markersize=4)
axes[1][1].plot(fl_round_losses, label="FL", color="green", marker="^", markersize=4)
axes[1][1].plot(fldp_losses, label="FL+DP", color="purple", marker="D", markersize=4)
axes[1][1].plot(niid_losses, label=f"FL Non-IID", color="brown", marker="x", markersize=4)
axes[1][1].set_title("Convergence Loss — Semua Skenario"); axes[1][1].legend(fontsize=7)

for ax_row in axes:
    for ax in ax_row: ax.set_xlabel(ax.get_xlabel() or "Round/Index")
plt.tight_layout(); plt.savefig("ablation_study.png", dpi=150, bbox_inches="tight"); plt.show()

print(f"Noise Multiplier optimal (accuracy): {noise_vals[exp1_acc.index(max(exp1_acc))]}")
print(f"Clipping Norm optimal (accuracy):    {norm_vals[exp2_acc.index(max(exp2_acc))]}")

### 8.2 Ablation: Jumlah Client, Local Epoch, Communication Rounds

In [ ]:
# ============================================================
# [Exp 3] Variasi Jumlah Client (num_clients)
# ============================================================
print("\n[Exp 3] Variasi Jumlah Client (noise=1.1, norm=1.0, rounds=5 tetap):")
client_counts = [2, 4]
exp3_acc, exp3_comm = [], []

for n_cli in client_counts:
    # Subset n_cli user pertama
    gm_tmp = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
    rnd_accs = []
    for rnd in range(1):  # 5 rounds
        gp = get_model_params(gm_tmp)
        cp, cs = [], []
        for uid in range(n_cli):
            lm = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
            set_model_params(lm, gp)
            up, _ = train_local(lm, user_datasets[uid]["X"], user_datasets[uid]["y"], epochs=2)
            cp.append(up); cs.append(len(user_datasets[uid]["X"]))
        set_model_params(gm_tmp, fedavg(cp, cs))
    acc, _, _ = evaluate_global(gm_tmp)
    comm_mb = n_cli * 2 * param_size_bytes / 1024 / 1024 * 5  # total 5 rounds
    exp3_acc.append(acc); exp3_comm.append(comm_mb)
    print(f"  n_clients={n_cli:<3} → Acc={acc:.4f}, Total Comm={comm_mb:.2f} MB")

# ============================================================
# [Exp 4] Variasi Local Epoch
# ============================================================
print("\n[Exp 4] Variasi Local Epoch (5 clients, 5 rounds, noise=1.1):")
local_epoch_vals = [1, 2]
exp4_acc, exp4_loss = [], []

for le in local_epoch_vals:
    gm_tmp = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
    rnd_loss_list = []
    for rnd in range(1):
        gp = get_model_params(gm_tmp)
        cp, cs, rl = [], [], 0.0
        for uid in range(5):  # 5 client
            lm = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
            set_model_params(lm, gp)
            up, ll = train_local(lm, user_datasets[uid]["X"], user_datasets[uid]["y"], epochs=le)
            cp.append(up); cs.append(len(user_datasets[uid]["X"])); rl += ll
        set_model_params(gm_tmp, fedavg(cp, cs))
        rnd_loss_list.append(rl / 5)
    acc, _, _ = evaluate_global(gm_tmp)
    exp4_acc.append(acc); exp4_loss.append(np.mean(rnd_loss_list))
    print(f"  local_epoch={le:<3} → Acc={acc:.4f}, Avg Loss={np.mean(rnd_loss_list):.4f}")

# ============================================================
# [Exp 5] Variasi Communication Rounds
# ============================================================
print("\n[Exp 5] Variasi Communication Rounds (5 clients, local_epoch=2):")
round_vals = [2]
exp5_acc, exp5_comm = [], []

gm_tmp = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
acc_per_round = []
comm_cumul_tmp = 0.0
for rnd in range(1, max(round_vals)+1):
    gp = get_model_params(gm_tmp); cp, cs = [], []
    for uid in range(5):
        lm = CNN_LSTM_Model(num_features, 256, len(user_map), dropout=0.3).to(device)
        set_model_params(lm, gp)
        up, _ = train_local(lm, user_datasets[uid]["X"], user_datasets[uid]["y"], epochs=2)
        cp.append(up); cs.append(len(user_datasets[uid]["X"]))
    set_model_params(gm_tmp, fedavg(cp, cs))
    comm_cumul_tmp += 5 * 2 * param_size_bytes / 1024 / 1024
    if rnd in round_vals:
        acc, _, _ = evaluate_global(gm_tmp)
        exp5_acc.append(acc); exp5_comm.append(comm_cumul_tmp)
        print(f"  round={rnd:<3} → Acc={acc:.4f}, Cumul Comm={comm_cumul_tmp:.2f} MB")


In [ ]:
# Visualisasi Ablation Tambahan (Exp 3, 4, 5)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Exp 3: Jumlah client vs Accuracy
axes[0].plot(client_counts, exp3_acc, "o-", color="teal", markersize=6)
axes[0].set_title("Akurasi vs Jumlah Client\n(rounds=5, local_epoch=2)")
axes[0].set_xlabel("Jumlah Client"); axes[0].set_ylabel("Accuracy")

ax0b = axes[0].twinx()
ax0b.plot(client_counts, exp3_comm, "s--", color="coral", markersize=5, alpha=0.7)
ax0b.set_ylabel("Total Comm Cost (MB)", color="coral")

# Exp 4: Local Epoch vs Accuracy & Loss
axes[1].plot(local_epoch_vals, exp4_acc, "o-", color="green", label="Accuracy", markersize=6)
ax1b = axes[1].twinx()
ax1b.plot(local_epoch_vals, exp4_loss, "s--", color="tomato", label="Avg Loss", markersize=5, alpha=0.8)
axes[1].set_title("Akurasi & Loss vs Local Epoch\n(5 clients, 5 rounds)")
axes[1].set_xlabel("Local Epoch"); axes[1].set_ylabel("Accuracy", color="green")
ax1b.set_ylabel("Avg Loss", color="tomato")

# Exp 5: Communication Rounds vs Accuracy
axes[2].plot(exp5_comm, exp5_acc, "o-", color="purple", markersize=6)
axes[2].set_title("Akurasi vs Cumulative Comm Rounds\n(5 clients, local_epoch=2)")
axes[2].set_xlabel("Cumulative Communication Cost (MB)")
axes[2].set_ylabel("Accuracy")
for i, (c, a, r) in enumerate(zip(exp5_comm, exp5_acc, round_vals)):
    axes[2].annotate(f"r={r}", (c, a), xytext=(5, 5), textcoords="offset points", fontsize=8)

plt.tight_layout()
plt.savefig("ablation_extra.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Ringkasan Ablation Study Lengkap ===")
print(f"[Noise Mult] Optimal: noise={noise_vals[exp1_acc.index(max(exp1_acc))]}")
print(f"[Clipping Norm] Optimal: norm={norm_vals[exp2_acc.index(max(exp2_acc))]}")
print(f"[Jumlah Client] Optimal: n_clients={client_counts[exp3_acc.index(max(exp3_acc))]}")
print(f"[Local Epoch] Optimal: local_epoch={local_epoch_vals[exp4_acc.index(max(exp4_acc))]}")
print(f"[Comm Rounds] Optimal: rounds={round_vals[exp5_acc.index(max(exp5_acc))]}")
print("\nTrade-off:")
print("  Utility: Noise multiplier rendah + local epoch lebih banyak = akurasi lebih tinggi")
print("  Privacy: Noise multiplier tinggi + clipping norm kecil = ε lebih kecil = privasi lebih kuat")
print("  Efisiensi: Lebih sedikit client & rounds = communication cost lebih rendah")
print("  Stabilitas: Local epoch terlalu besar = client drift meningkat (terutama pada Non-IID)")


## Level 9: Advanced Leakage Attack — Deep Leakage from Gradients (DLG)

**Referensi:** Zhu et al. (2019) *Deep Leakage from Gradients*

Penyerang dengan akses ke **gradien model** mencoba merekonstruksi **data training asli** via optimasi iteratif.
Metrik: reconstruction MSE, leakage success rate, FL tanpa DP vs FL+DP.

In [ ]:
def compute_grads(m, X_sample, y_sample, crit_fn):
    m.zero_grad()
    X_t = torch.tensor(X_sample, dtype=torch.float32).unsqueeze(0).to(device)
    y_t = torch.tensor([y_sample], dtype=torch.long).to(device)
    crit_fn(m(X_t), y_t).backward()
    return [p.grad.clone() for p in m.parameters() if p.grad is not None]

def dlg_attack(m, real_grads, n_iter=5, lr=0.01):
    """Simplified DLG: minimalkan ||∇(dummy) - ∇(real)||²"""
    dummy_x = torch.randn(1, WINDOW_SIZE, num_features, requires_grad=True, device=device)
    dummy_y = torch.randint(0, len(user_map), (1,), device=device)
    opt_dlg = torch.optim.LBFGS([dummy_x], lr=lr)
    crit_dlg = nn.CrossEntropyLoss()
    history = []
    for _ in range(n_iter):
        def closure():
            opt_dlg.zero_grad()
            out = m(dummy_x); loss = crit_dlg(out, dummy_y)
            dg = torch.autograd.grad(loss, m.parameters(), create_graph=True, allow_unused=True)
            gl = sum(((g-r.detach())**2).sum() for g,r in zip(dg, real_grads) if g is not None)
            gl.backward(); history.append(gl.item()); return gl
        opt_dlg.step(closure)
    return dummy_x.detach().cpu().numpy(), history

crit_dlg = nn.CrossEntropyLoss()
real_X_sample = X_test[0]; real_y_sample = int(y_test[0])
print(f"DLG target: rekonstruksi data {user_names[real_y_sample]} dari gradien")

# DLG pada FL (tanpa DP)
print("\n[DLG] Menyerang FL model (tanpa DP)...")
m_fl_atk = copy.deepcopy(global_model)
m_fl_atk.train()
for p in m_fl_atk.parameters(): p.requires_grad_(True)
rg_fl = compute_grads(m_fl_atk, real_X_sample, real_y_sample, crit_dlg)
rec_fl, hist_fl = dlg_attack(m_fl_atk, rg_fl, n_iter=5, lr=0.01)
mse_fl = np.mean((rec_fl[0] - real_X_sample)**2)
print(f"  MSE (FL tanpa DP): {mse_fl:.6f}")

# DLG pada FL+DP
print("\n[DLG] Menyerang FL+DP model...")
m_fldp_atk = copy.deepcopy(global_model_fldp._module if hasattr(global_model_fldp,"_module") else global_model_fldp)
m_fldp_atk.train()
for p in m_fldp_atk.parameters(): p.requires_grad_(True)
rg_fldp = compute_grads(m_fldp_atk, real_X_sample, real_y_sample, crit_dlg)
rec_fldp, hist_fldp = dlg_attack(m_fldp_atk, rg_fldp, n_iter=5, lr=0.01)
mse_fldp = np.mean((rec_fldp[0] - real_X_sample)**2)
print(f"  MSE (FL+DP): {mse_fldp:.6f}")

THRESH = 0.1
leak_fl   = "BOCOR" if mse_fl   < THRESH else "Aman"
leak_fldp = "BOCOR" if mse_fldp < THRESH else "Aman"
print(f"\n  FL tanpa DP: {leak_fl} (MSE={mse_fl:.6f})")
print(f"  FL + DP:     {leak_fldp} (MSE={mse_fldp:.6f})")
print(f"  DP meningkatkan MSE rekonstruksi sebesar {((mse_fldp/max(mse_fl,1e-9))-1)*100:.0f}%")

### 9.1 Visualisasi DLG Attack

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(hist_fl[:50], label="FL (tanpa DP)", color="tomato", lw=2)
axes[0].plot(hist_fldp[:50], label="FL+DP", color="green", lw=2)
axes[0].set_title("DLG Gradient Distance per Iterasi"); axes[0].legend()
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Gradient Distance Loss")

feat_idx = 4  # speed
axes[1].plot(real_X_sample[:,feat_idx], label="Asli", color="steelblue", lw=2)
axes[1].plot(rec_fl[0,:,feat_idx], label=f"Rekonstruksi FL (MSE={mse_fl:.5f})", color="tomato", linestyle="--")
axes[1].set_title("Rekonstruksi Speed (FL tanpa DP)"); axes[1].legend()

axes[2].plot(real_X_sample[:,feat_idx], label="Asli", color="steelblue", lw=2)
axes[2].plot(rec_fldp[0,:,feat_idx], label=f"Rekonstruksi FL+DP (MSE={mse_fldp:.5f})", color="purple", linestyle="--")
axes[2].set_title("Rekonstruksi Speed (FL+DP)"); axes[2].legend()

plt.tight_layout(); plt.savefig("dlg_results.png", dpi=150, bbox_inches="tight"); plt.show()

## Level 10: Explainability dan Threat Modeling

### 10A: SHAP Analysis
Identifikasi fitur pola mouse yang paling berpengaruh pada keputusan autentikasi CNN-LSTM.

In [ ]:
import shap

print("=== Level 10A: SHAP Feature Importance ===")
background_X = torch.tensor(X_train[:2], dtype=torch.float32).to(device)
explain_X    = torch.tensor(X_test[:1], dtype=torch.float32).to(device)
model.eval()

e = shap.GradientExplainer(model, background_X)
shap_vals = e.shap_values(explain_X)

# Rata-rata |SHAP| per fitur
shap_importance = np.abs(np.array(shap_vals)).mean(axis=(0,1,2))
sorted_idx = np.argsort(shap_importance)[::-1]
sorted_feats = [FEATURE_NAMES[i] for i in sorted_idx]
sorted_imp   = shap_importance[sorted_idx]

print("Fitur mouse paling berpengaruh:")
for rank, (feat, imp) in enumerate(zip(sorted_feats, sorted_imp), 1):
    print(f"  {rank:2d}. {feat:<15} {imp:.6f}")

plt.figure(figsize=(10, 6))
colors_shap = plt.cm.RdYlGn(np.linspace(0.8, 0.2, len(sorted_feats)))
plt.barh(sorted_feats[::-1], sorted_imp[::-1], color=colors_shap)
plt.xlabel("Mean |SHAP Value| (Feature Importance)")
plt.title("SHAP Feature Importance — CNN-LSTM Behavioral Biometrics\n"
          "(Merah = paling berpengaruh = paling sensitif terhadap privasi)")
plt.tight_layout(); plt.savefig("shap_importance.png", dpi=150, bbox_inches="tight"); plt.show()

print(f"\nFitur paling sensitif terhadap privasi: {sorted_feats[0]}, {sorted_feats[1]}, {sorted_feats[2]}")

### 10B: Threat Model Analysis

In [ ]:
import pandas as pd

print("=== Level 10B: Threat Model Analysis ===")

threat_model = {
    "Ancaman": ["Honest-but-Curious Server","Malicious Client","External Attacker",
                "Insider Attack","Gradient Leakage Risk","Membership Inference Risk"],
    "Baseline ML": ["KRITIS: server lihat semua data","N/A","TINGGI: model bisa di-query",
                    "KRITIS: data terpusat","N/A","TINGGI: model overfit"],
    "DP-SGD": ["SEDANG: data masih terpusat, model lebih aman","N/A","RENDAH: noise cegah re-id",
               "SEDANG: data masih terpusat","RENDAH: gradien ber-noise","RENDAH: DP batasi MIA"],
    "FL": ["SEDANG: server hanya terima gradient","SEDANG: gradient poisoning mungkin",
           "SEDANG: data lokal aman, model bisa di-query","RENDAH: data tidak keluar perangkat",
           "TINGGI: DLG bisa invert gradien!","SEDANG: FL kurangi overfit"],
    "FL+DP": ["RENDAH: server tidak lihat data & gradien ber-noise","SEDANG: perlu FedProx/Krum",
              "SANGAT RENDAH: data lokal+noise gradien","RENDAH: kombinasi terkuat",
              "SANGAT RENDAH: DP buat DLG gagal","SANGAT RENDAH: pertahanan terkuat MIA"]
}

df_threat = pd.DataFrame(threat_model)
print(df_threat.to_string(index=False))

# Heatmap risiko
risk_map = {"SANGAT RENDAH":1,"RENDAH":2,"SEDANG":3,"TINGGI":4,"KRITIS":5,"N/A":0}
risk_matrix = []
for col in ["Baseline ML","DP-SGD","FL","FL+DP"]:
    col_risk = [risk_map.get(next((k for k in risk_map if k in v), "SEDANG"), 3) for v in df_threat[col]]
    risk_matrix.append(col_risk)
risk_matrix = np.array(risk_matrix).T

plt.figure(figsize=(12, 7))
im = plt.imshow(risk_matrix, cmap="RdYlGn_r", vmin=0, vmax=5, aspect="auto")
plt.colorbar(im, label="Risk Level (1=Sangat Rendah, 5=Kritis, 0=N/A)")
plt.xticks(range(4), ["Baseline ML","DP-SGD","FL","FL+DP"], fontsize=11)
plt.yticks(range(6), threat_model["Ancaman"], fontsize=10)
plt.title("Threat Model Heatmap — Behavioral Biometrics System")
risk_labels = {0:"N/A",1:"Sangat Rendah",2:"Rendah",3:"Sedang",4:"Tinggi",5:"Kritis"}
for i in range(6):
    for j in range(4):
        plt.text(j, i, risk_labels[risk_matrix[i,j]], ha="center", va="center",
                 fontsize=7, fontweight="bold", color="black")
plt.tight_layout(); plt.savefig("threat_model.png", dpi=150, bbox_inches="tight"); plt.show()

print("\n" + "="*60)
print("IMPLEMENTASI LEVEL 1-10 SELESAI!")
print("="*60)
print(f"Fitur paling sensitif: {sorted_feats[:3]}")
print("Skenario paling aman: FL + DP")
print("Skenario paling rentan: Baseline ML")
print("Rekomendasi deployment: FL+DP dengan ε ≤ 5.0")